In [ ]:
"""
Geo Snap – Task 1B: Multispectral Classification
Custom CNN from scratch | in_channels=13 (+ 3 spectral indices = 16)
Dataset: EuroSATallBands (.tif, 64x64x13)
"""

import os, json, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import tifffile
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# ── CONFIG ───────────────────────────────────────────────────────────────────
BASE_DIR   = Path("EuroSAT_Dataset")          # ← set to your dataset root
CKPT_DIR   = Path("checkpoints_ms"); CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR    = Path("outputs_ms");     OUT_DIR.mkdir(exist_ok=True)

BATCH      = 128
EPOCHS     = 60
LR         = 3e-4
NUM_WORKERS= 4
SEED       = 42

CLASS_NAMES = ["AnnualCrop","Forest","HerbaceousVegetation","Highway",
               "Industrial","Pasture","PermanentCrop","Residential","River","SeaLake"]

# EuroSAT per-band mean/std (bands: B01-B12 order in .tif)
MEAN = np.array([1370.19,1184.35,1120.20,1136.37,954.68,1757.57,
                 2002.07,2119.72,2186.73,810.25,22.89,1793.05,1219.72],np.float32)
STD  = np.array([633.15,650.09,594.58,630.31,505.09,749.17,
                 836.09,848.08,845.78,601.01,37.24,740.28,616.96],np.float32)

# ── UTILS ────────────────────────────────────────────────────────────────────
def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def ndvi_ndwi_nbr(img):
    """img: (13,H,W) raw → returns (3,H,W) indices"""
    e  = 1e-6
    nir,red,grn,swir = img[7],img[3],img[2],img[11]
    ndvi = (nir-red)/(nir+red+e)
    ndwi = (grn-nir)/(grn+nir+e)
    nbr  = (nir-swir)/(nir+swir+e)
    return np.stack([ndvi,ndwi,nbr],0).clip(-1,1).astype(np.float32)

def load_tif(path):
    img = tifffile.imread(str(path)).astype(np.float32)
    if img.shape[-1] == 13: img = img.transpose(2,0,1)
    return img  # (13,64,64)

# ── DATASET ──────────────────────────────────────────────────────────────────
class MSDataset(Dataset):
    def __init__(self, root, split, augment=False):
        self.augment = augment
        split_dir = root/"EuroSATallBands"/split
        self.samples = []
        for i,cls in enumerate(CLASS_NAMES):
            for f in (split_dir/cls).glob("*.tif"):
                self.samples.append((f, i))
        print(f"[{split}] {len(self.samples)} images")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = load_tif(path)
        idx_ch = ndvi_ndwi_nbr(img)
        img = (img - MEAN[:,None,None]) / (STD[:,None,None]+1e-6)
        img = np.concatenate([img, idx_ch], 0)  # (16,64,64)
        if self.augment:
            if random.random()>.5: img = img[:,:,::-1].copy()
            if random.random()>.5: img = img[:,::-1,:].copy()
            k = random.randint(0,3)
            if k: img = np.rot90(img,k,(1,2)).copy()
        return torch.from_numpy(img.copy()), label

class TestDataset(Dataset):
    def __init__(self, test_dir):
        self.paths = sorted(Path(test_dir).glob("*.tif"))
        print(f"[test] {len(self.paths)} images")
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = load_tif(self.paths[idx])
        idx_ch = ndvi_ndwi_nbr(img)
        img = (img - MEAN[:,None,None]) / (STD[:,None,None]+1e-6)
        img = np.concatenate([img, idx_ch], 0)
        return torch.from_numpy(img.copy()), self.paths[idx].stem

# ── MODEL ────────────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c,out_c,3,stride,1,bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True),
            nn.Conv2d(out_c,out_c,3,1,1,bias=False),     nn.BatchNorm2d(out_c))
        self.skip = nn.Sequential(
            nn.Conv2d(in_c,out_c,1,stride,bias=False), nn.BatchNorm2d(out_c)
        ) if stride!=1 or in_c!=out_c else nn.Identity()
        self.relu = nn.ReLU(True)
    def forward(self,x): return self.relu(self.conv(x)+self.skip(x))

class SpectralSE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                nn.Linear(c,c//r), nn.ReLU(True),
                                nn.Linear(c//r,c), nn.Sigmoid())
    def forward(self,x): return x * self.fc(x).view(x.size(0),x.size(1),1,1)

class MultiSpectralCNN(nn.Module):
    def __init__(self, in_ch=16, nc=10):
        super().__init__()
        def cbr(i,o,s=1): return nn.Sequential(
            nn.Conv2d(i,o,3,s,1,bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
        self.stem   = nn.Sequential(cbr(in_ch,64), cbr(64,64))
        self.se     = SpectralSE(64)
        self.stage1 = nn.Sequential(ResBlock(64,64),   ResBlock(64,64))
        self.stage2 = nn.Sequential(ResBlock(64,128,2), ResBlock(128,128))
        self.stage3 = nn.Sequential(ResBlock(128,256,2),ResBlock(256,256))
        self.stage4 = nn.Sequential(ResBlock(256,512,2),ResBlock(512,512))
        self.head   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                    nn.Dropout(0.4), nn.Linear(512,nc))
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.Linear): nn.init.xavier_normal_(m.weight)
    def forward(self,x):
        return self.head(self.stage4(self.stage3(self.stage2(self.stage1(self.se(self.stem(x)))))))

# ── TRAIN / EVAL ─────────────────────────────────────────────────────────────
def one_epoch(model, loader, crit, opt, scaler, device):
    model.train()
    loss_sum,correct,n = 0,0,0
    for x,y in loader:
        x,y = x.to(device,non_blocking=True), y.to(device)
        opt.zero_grad(set_to_none=True)
        with autocast():
            out  = model(x)
            loss = crit(out,y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)
        scaler.step(opt); scaler.update()
        loss_sum += loss.item()*x.size(0)
        correct  += (out.argmax(1)==y).sum().item(); n += x.size(0)
    return loss_sum/n, correct/n

@torch.no_grad()
def evaluate(model, loader, crit, device):
    model.eval()
    loss_sum,correct,n = 0,0,0
    all_p,all_y = [],[]
    for x,y in loader:
        x,y = x.to(device,non_blocking=True), y.to(device)
        with autocast():
            out  = model(x); loss = crit(out,y)
        loss_sum += loss.item()*x.size(0)
        p = out.argmax(1)
        correct += (p==y).sum().item(); n += x.size(0)
        all_p.extend(p.cpu().numpy()); all_y.extend(y.cpu().numpy())
    return loss_sum/n, correct/n, all_p, all_y

# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    seed_all(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if device.type=='cuda' else 'N/A'}")

    train_ds = MSDataset(BASE_DIR,"train",augment=True)
    val_ds   = MSDataset(BASE_DIR,"val",  augment=False)
    train_dl = DataLoader(train_ds,BATCH,shuffle=True, num_workers=NUM_WORKERS,pin_memory=True,drop_last=True,persistent_workers=True)
    val_dl   = DataLoader(val_ds,  BATCH*2,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)

    model  = MultiSpectralCNN(in_ch=16, nc=10).to(device)
    crit   = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt    = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = GradScaler()

    best_acc, best_path = 0., CKPT_DIR/"best_ms.pth"
    patience, no_imp   = 12, 0
    hist = {"tl":[],"ta":[],"vl":[],"va":[]}

    for ep in range(1, EPOCHS+1):
        t0 = time.time()
        tl,ta = one_epoch(model,train_dl,crit,opt,scaler,device)
        vl,va,preds,labels = evaluate(model,val_dl,crit,device)
        sched.step()
        hist["tl"].append(tl); hist["ta"].append(ta)
        hist["vl"].append(vl); hist["va"].append(va)
        mark = ""
        if va > best_acc:
            best_acc = va; no_imp = 0
            torch.save(model.state_dict(), best_path); mark = " ✓ saved"
        else:
            no_imp += 1
        print(f"Ep {ep:3d}/{EPOCHS} | lr {opt.param_groups[0]['lr']:.1e} | "
              f"tr {ta*100:.2f}% {tl:.4f} | val {va*100:.2f}% {vl:.4f} | "
              f"{time.time()-t0:.0f}s{mark}")
        if no_imp >= patience:
            print(f"Early stop at ep {ep}"); break

    # ── Plots ─────────────────────────────────────────────────────────────────
    ep_range = range(1, len(hist["tl"])+1)
    fig,ax = plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(ep_range,hist["tl"],label="train"); ax[0].plot(ep_range,hist["vl"],label="val")
    ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(ep_range,hist["ta"],label="train"); ax[1].plot(ep_range,hist["va"],label="val")
    ax[1].set_title("Accuracy"); ax[1].legend()
    plt.savefig(OUT_DIR/"curves_ms.png",dpi=150); plt.close()

    # ── Final eval ────────────────────────────────────────────────────────────
    model.load_state_dict(torch.load(best_path, map_location=device))
    _,acc,preds,labels = evaluate(model,val_dl,crit,device)
    print(f"\nBest val acc: {acc*100:.2f}%")
    print(classification_report(labels,preds,target_names=CLASS_NAMES,digits=4))

    cm = confusion_matrix(labels,preds)
    fig,ax = plt.subplots(figsize=(10,8))
    im = ax.imshow(cm,cmap="Blues")
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASS_NAMES,rotation=45,ha="right")
    ax.set_yticks(range(10)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(10):
        for j in range(10):
            ax.text(j,i,cm[i,j],ha="center",va="center",fontsize=7,
                    color="white" if cm[i,j]>cm.max()/2 else "black")
    ax.set_title("Confusion Matrix – Multispectral"); plt.tight_layout()
    plt.savefig(OUT_DIR/"cm_ms.png",dpi=150); plt.close()

    # ── Inference ─────────────────────────────────────────────────────────────
    test_dir = BASE_DIR/"EuroSATallBands_test_flat"
    if test_dir.exists():
        test_dl = DataLoader(TestDataset(test_dir), BATCH*2, num_workers=NUM_WORKERS,pin_memory=True)
        model.eval(); rows=[]
        with torch.no_grad():
            for x,fnames in test_dl:
                x = x.to(device)
                with autocast(): preds = model(x).argmax(1).cpu().numpy()
                for name,pred in zip(fnames,preds):
                    rows.append({"img_id":name,"predicted_label":int(pred)})
        pd.DataFrame(rows).to_csv(OUT_DIR/"ms_predictions.csv",index=False)
        print(f"Saved {len(rows)} predictions → {OUT_DIR/'ms_predictions.csv'}")
    else:
        print("Test dir not found, skipping inference.")

if __name__ == "__main__":
    main()